# 01 — Explorar e limpar: dormidas no Algarve

Primeiro olhar aos dados brutos (`dados/dormidas_algarve_bruto.csv`), registo de cada regra de limpeza com contagens e verificação final antes da análise.


In [ ]:
import pandas as pd

df = pd.read_csv("../dados/dormidas_algarve_bruto.csv", sep=";", decimal=",", encoding="utf-8")
print("Forma inicial:", df.shape)
print("\nTipos:")
print(df.dtypes)
print("\nVazios iniciais:")
print(df.isna().sum())
df.head(10)


## Regras 1 e 2: normalizar municípios e remover duplicados

- `municipio`: tirar espaços nas pontas para não separar o mesmo concelho em duas categorias.
- Chave única: `(municipio, ano, mes)` — incluir `ano` evita apagar os outros anos.


In [ ]:
# Regra 1: remover espaços invisíveis nas pontas dos nomes dos municípios
df["municipio"] = df["municipio"].str.strip()

# Regra 2: remover linhas duplicadas pela chave (municipio, ano, mes), mantendo a primeira
antes = len(df)
df = df.drop_duplicates(subset=["municipio", "ano", "mes"], keep="first").copy()
removidas_duplicadas = antes - len(df)
print("Duplicadas removidas:", removidas_duplicadas)


## Regra 3: converter `dormidas` para número e tirar linhas sem valor

Valores marcados como `n.d.` ou vazios passam a `NaN` e são removidos (trocar por 0 distorceria as médias mensais).


In [ ]:
# Regra 3: converter dormidas para número (troca vírgula decimal por ponto; "n.d." passa a NaN)
# e remover linhas sem valor numérico (não permitem somar dormidas reais).
s_num = df["dormidas"].astype(str).str.replace(",", ".", regex=False)
df["dormidas"] = pd.to_numeric(s_num, errors="coerce")

antes = len(df)
df = df.dropna(subset=["dormidas"]).copy()
removidas_sem_valor = antes - len(df)
df["dormidas"] = df["dormidas"].round().astype(int)

print("Sem valor ('n.d.' ou vazio) removidas:", removidas_sem_valor)
print("\n--- Relatório de limpeza ---")
print(f"Entraram: {len(df) + removidas_duplicadas + removidas_sem_valor}")
print(f"Duplicadas removidas: {removidas_duplicadas}")
print(f"Sem valor removidas: {removidas_sem_valor}")
print(f"Ficaram: {len(df)}")


## Verificação final e exportação

Confirmar chave única, tipos corretos, zero nulos e gravar em `../dados/dormidas_algarve_limpo.csv` sem tocar no ficheiro original.


In [ ]:
# Ponto de verificação: os 5 sinais de que os dados estão prontos
assert df.duplicated(subset=["municipio", "ano", "mes"]).sum() == 0
assert pd.api.types.is_integer_dtype(df["dormidas"])
assert df["municipio"].nunique() == 16
assert df.isna().sum().sum() == 0

df.to_csv("../dados/dormidas_algarve_limpo.csv", index=False, encoding="utf-8")
print("Ficheiro limpo guardado em ../dados/dormidas_algarve_limpo.csv")
